# Which warehouse serves each customer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ula-uab/lscm-decision-making/blob/main/cases/warehouse_allocation/notebooks/warehouse_allocation.ipynb)

A company ships pallets from two warehouses, W1 and W2, to four customers, C1 to C4. Each week it decides which warehouse serves each customer. This is the running example of T1, and this notebook lets you play with it:

1. **Experience or a model?** You plan by hand, then compare with the rule of thumb and with the model.
2. **How many plans are there?** Few here, far too many in a real network.
3. **Two objectives.** Cost and delivery time pull in different directions.
4. **Uncertainty.** The plan rests on a forecast, and the forecast can be wrong.

**How to use it.** Run the cells in order, with ▶ or *Shift + Enter*. Some cells have a form (drop-down lists and sliders): change the values and run the cell again. Outside Colab the form is a line such as `C1 = "W1"`: edit the value and run the cell.

All the data and results are explained in the [document of the example](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/warehouse_allocation/warehouse_allocation.md). Running the notebook is optional: it is support material, not part of the assessment.

In [ ]:
# @title Setup (run this cell first) { display-mode: "form" }
# In Google Colab, install the example from GitHub (it takes a few seconds).
# On your own computer, install the repository environment instead (see README).
import sys
if "google.colab" in sys.modules:
    %pip install -q "git+https://github.com/ula-uab/lscm-decision-making#subdirectory=cases/warehouse_allocation"
import warehouses as wa

## 1. The situation

Each warehouse can ship a limited number of pallets per week, and each customer has a forecast demand. Shipping a pallet has a cost and takes some days, which depend on the warehouse and the customer. **Each customer is served entirely from one warehouse**: its order is not split.

The map is a sketch, not to scale: each customer is drawn nearer the warehouse that is cheaper and faster for it. Each link shows the cost per pallet and the delivery time.

In [ ]:
# @title Data and map { display-mode: "form" }
wa.show_data()

## 2. Be the planner

A **plan** says which warehouse serves each customer. Choose one below and run the cell. You will see:

- on the map, which warehouse serves each customer;
- the **load** of each warehouse, the pallets of the customers it serves, against its capacity: the load of $w$ is $\sum_{c} d_c\, x_{wc}$, where $d_c$ is the demand of customer $c$ and $x_{wc}$ is 1 if $w$ serves $c$ and 0 otherwise. A plan is **feasible** if no load goes over its capacity $K_w$;
- the **cost** of the plan: pallets × cost per pallet, added over all customers, $\sum_{w}\sum_{c} k_{wc}\, d_c\, x_{wc}$.

The plan proposed at first sends everything from W1. Change it and try to find a cheap plan that fits.

In [ ]:
# @title Your plan { display-mode: "form" }
C1 = "W1"  # @param ["W1", "W2"]
C2 = "W1"  # @param ["W1", "W2"]
C3 = "W1"  # @param ["W1", "W2"]
C4 = "W1"  # @param ["W1", "W2"]
wa.check_plan({"C1": C1, "C2": C2, "C3": C3, "C4": C4})

## 3. Deciding from experience versus deciding with a model

**Rule of thumb.** An experienced planner serves each customer from its nearest warehouse: W1 for C1, C2 and C3, W2 for C4. It is also the cheapest and fastest link for each one. Does it fit?

**Diverting by hand.** When the nearest warehouse is full, a usual fix is to take the customers one by one, in order C1, C2, C3, C4, and send each one to its nearest warehouse if there is room left, or to the other one if not. This gives **plan M**.

In [ ]:
# @title Rule of thumb and plan M { display-mode: "form" }
wa.rule_of_thumb()

### Challenge: beat plan M

Plan M fits and costs 465 €/week. Can you find a feasible plan that costs less? The form starts from plan M.

In [ ]:
# @title Beat plan M { display-mode: "form" }
C1 = "W1"  # @param ["W1", "W2"]
C2 = "W1"  # @param ["W1", "W2"]
C3 = "W2"  # @param ["W1", "W2"]
C4 = "W2"  # @param ["W1", "W2"]
wa.challenge({"C1": C1, "C2": C2, "C3": C3, "C4": C4})

### The model

The same problem, written in mathematical form. For each warehouse $w$ and customer $c$, the decision is $x_{wc} = 1$ if $w$ serves $c$, and $0$ otherwise. The model seeks the lowest total shipping cost:

$$
\begin{aligned}
\min \quad & \sum_{w \in W} \sum_{c \in C} k_{wc}\, d_c\, x_{wc} \\
\text{s.t.} \quad & \sum_{w \in W} x_{wc} = 1 && \forall c \in C && \text{(each customer from one warehouse)}\\
& \sum_{c \in C} d_c\, x_{wc} \le K_w && \forall w \in W && \text{(capacity)}\\
& x_{wc} \in \{0, 1\} && \forall w \in W,\ \forall c \in C
\end{aligned}
$$

The cell below writes these lines in Python with the library PuLP, and the solver HiGHS finds the best plan. The code is in the file `model.py` of the example, function `solve`: each line of the model is one line of code.

In [ ]:
# @title Solve the model { display-mode: "form" }
plan_A = wa.solve_model()

The model keeps C3 in W1 and diverts C2 instead. The rule of thumb diverts whichever customer happens to come last, not the one that is cheapest to divert. Did you find plan A in the challenge?

## 4. How many plans are there?

Each of the 4 customers can be served by either of the 2 warehouses: $2 \times 2 \times 2 \times 2 = 2^4 = 16$ plans. Only 5 of them fit in the warehouses.

In [ ]:
# @title All 16 plans { display-mode: "form" }
wa.all_plans_table()

Sixteen plans can be checked by hand. With $m$ warehouses and $n$ customers there are $m^n$ plans. The table below assumes a computer that checks a billion plans per second, which is generous, and compares the time with the age of the universe, about 13.8 billion years (Planck Collaboration, 2020, *Astronomy & Astrophysics* 641, A6).

In [ ]:
# @title The number of plans grows very fast { display-mode: "form" }
wa.explosion_table()

This is why checking every plan (*brute force*) is not a method, and why T2 studies optimisation methods. A solver does not list the plans: it finds the best one by reasoning on the model. Here both ways give the same plan.

In [ ]:
# @title Brute force and solver { display-mode: "form" }
wa.compare_methods()

## 5. Two objectives: cost and delivery time

So far the plan has been judged only by its cost. The company also wants its customers served fast, and the cheapest plan is not the fastest.

### The second objective: average delivery time

A pallet of customer $c$ sent from warehouse $w$ takes $t_{wc}$ days to arrive (Table 4). The **average delivery time** of a plan is the number of days that a pallet takes on average:

$$
T(x) = \frac{\displaystyle\sum_{w \in W} \sum_{c \in C} t_{wc}\, d_c\, x_{wc}}{\displaystyle\sum_{c \in C} d_c}
$$

- In the numerator, $t_{wc}\, d_c\, x_{wc}$ counts the **pallet-days** of customer $c$: its $d_c$ pallets, each taking $t_{wc}$ days, if $w$ serves it ($x_{wc} = 1$), and nothing otherwise.
- The denominator is the total number of pallets, $\sum_c d_c = 130$.
- A large customer served slowly weighs more than a small one.

For plan A (C1 and C3 from W1, C2 and C4 from W2):

$$
T = \frac{40 \cdot 1 + 30 \cdot 4 + 35 \cdot 1 + 25 \cdot 1}{130} = \frac{220}{130} = 1.69 \text{ days}
$$

### The problem with two objectives

The model is the one of §3, with a second objective. The decisions and the constraints do not change:

$$
\begin{aligned}
\min \quad & Z_1(x) = \sum_{w \in W} \sum_{c \in C} k_{wc}\, d_c\, x_{wc} && \text{(cost, €/week)}\\
\min \quad & Z_2(x) = \frac{1}{\sum_{c} d_c} \sum_{w \in W} \sum_{c \in C} t_{wc}\, d_c\, x_{wc} && \text{(average delivery time, days)}\\
\text{s.t.} \quad & \sum_{w \in W} x_{wc} = 1 && \forall c \in C\\
& \sum_{c \in C} d_c\, x_{wc} \le K_w && \forall w \in W\\
& x_{wc} \in \{0, 1\} && \forall w \in W,\ \forall c \in C
\end{aligned}
$$

With two objectives there is usually no plan that is best on both. Solving the problem has two steps:

1. **Discard the plans that no one should choose**, and keep the Pareto-optimal ones.
2. **Choose among those**, which needs the company to say how much one objective is worth against the other.

### Step 1: Pareto-optimal plans

A plan **beats** another if it is no worse on both criteria and better on at least one. A feasible plan is **Pareto-optimal** if no other feasible plan beats it. A plan that is beaten should never be chosen: there is another plan that is at least as cheap and at least as fast.

In the figure, the x-axis is the average delivery time and the y-axis the cost: **down is cheaper, left is faster**. A plan beats another if it lies below and to the left of it. The table says, for each feasible plan, which plans beat it.

In [ ]:
# @title Pareto-optimal plans { display-mode: "form" }
wa.pareto()

- **A** is Pareto-optimal because no feasible plan is cheaper: nothing lies below it.
- **B** is Pareto-optimal because no feasible plan is faster: nothing lies to its left.
- **M**, the plan diverted by hand, is beaten by B, which is cheaper and faster (green arrow). **D** and **E** are beaten by A, among others.
- The crosses do not count: the cheapest one (290 €/week, 1.00 days) would beat every plan, but it does not fit in the warehouses.

So the choice is between A (cheaper) and B (faster). Going from A to B costs $450 - 320 = 130$ €/week more and shortens the average delivery by $1.69 - 1.31 = 0.385$ days (exactly $50/130$). Is it worth it? The model cannot say: it depends on what a faster delivery is worth to the company.

### Step 2: choosing with a price for time

One way to choose is to give time a price. Let $V$ be **what the company would pay, in euros per week, to make the average delivery one day shorter**. It is measured in **€/week per day**: euros per week (like the cost), for each day of average delivery time that is saved. For example, $V = 100$ means that shortening the average delivery by one day is worth 100 €/week to the company, and shortening it by half a day is worth 50 €/week. The value comes from the company: fewer complaints, sales that are not lost, smaller stocks at the customers.

With $V$, the two objectives become one, in €/week:

$$
\min \quad Z(x) = Z_1(x) + V \cdot Z_2(x) \qquad \text{(cost + V × average delivery time)}
$$

with the same constraints. This is called the **weighted sum** method: $V$ is the weight of the delivery time. The model is of the same kind as the one of §3, and the same solver solves it. For plan A with $V = 100$: $Z = 320 + 100 \times 1.69 = 489$ €/week; for plan B: $Z = 450 + 100 \times 1.31 = 581$ €/week. With $V = 100$, A is better.

**Where the choice changes.** B saves 0.385 days for 130 €/week more, so each day saved costs $130 / 0.385 = 338$ €/week. If one day is worth less than 338 €/week to the company ($V < 338$), the saving is not worth its cost and A is chosen. If it is worth more ($V > 338$), B is chosen. At $V = 338$ both plans have the same total.

**The dotted line in the figure.** It joins all the points with the same total as the chosen plan: $\text{cost} + V \times \text{time} = Z$, that is, $\text{cost} = Z - V \times \text{time}$. It is a straight line with **slope $-V$**: along it, one day more of delivery time is compensated by $V$ euros less of cost. Points below the line have a lower total, points above it a higher one. The best plan is the feasible plan on the lowest such line: no feasible plan lies below it. When $V$ grows the line gets steeper and time weighs more: above 338 the lowest line goes through B instead of A.

Move the slider: try values below and above 338.

In [ ]:
# @title How much is a day worth? { display-mode: "form" }
value_of_a_day = 100  # @param {type:"slider", min:0, max:1000, step:10}
wa.two_objectives(value_of_a_day)

Plans M, D and E are never chosen, whatever the value of $V$: each is beaten by a plan that is cheaper and faster, so that plan always has a lower total. The weighted sum is one of several ways of choosing between objectives; another is to minimise the cost with a limit on the delivery time. T3-T comes back to them.

## 6. Uncertainty: the forecast can be wrong

The plan uses the forecast demand. The forecast of C3 is the average of its last four weeks of orders (a 4-week moving average). Its error is orders − forecast, and it is summarised with:

- **MAE** (mean absolute error): the average size of the error, in pallets/week;
- **MAPE** (mean absolute percentage error): the same, as a percentage of the orders;
- **bias**: the average error with its sign; close to zero means the forecast is not systematically too high or too low.

In [ ]:
# @title Orders and forecast of C3 { display-mode: "form" }
wa.show_forecast()

### What if the error is large?

In week 13 the forecast of C3 is 35 pallets, but C3 orders 46. Does plan A still fit? Try other orders with the slider. The cell also **re-plans**: it solves the model again with the new order. In practice the plan is recomputed every week as new orders arrive, which is called planning with a rolling horizon (T3-P).

In [ ]:
# @title A large error in week 13 { display-mode: "form" }
c3_order = 46  # @param {type:"slider", min:25, max:55, step:1}
wa.large_error(c3_order)

### What does a safety margin cost?

One way to protect the plan is to keep part of the capacity in reserve: with a 10 % reserve, W1 plans with 72 pallets instead of 80. The model then chooses the best plan with less capacity, and the margin costs money. The second figure shows whether that plan copes with the real order of C3 in week 13.

Try a reserve of 10 % and then of 7 %. Which one copes with the 46 pallets of C3? Why is more reserve not always more protection?

In [ ]:
# @title A safety margin { display-mode: "form" }
reserve_percent = 10  # @param {type:"slider", min:0, max:15, step:1}
c3_order = 46  # @param {type:"slider", min:25, max:55, step:1}
wa.safety_margin(reserve_percent, c3_order)

## 7. Plans under uncertain demand

Section 6 looked at one week: the forecast of C3 fails in week 13. Here a plan is kept for several weeks while the orders of two customers change. You can check any feasible plan week by week, measure the forecast error of C1 and C3, and compute what each plan costs over a period when a pallet that is not served has a cost.

The assumptions:

1. The plan (which warehouse serves each customer) is kept for $N$ weeks, with no changes.
2. Each customer is served from one warehouse. The pallets above the capacity of a warehouse in a week are not served: they are lost, and each one costs $p$ € to the company.
3. Each week is independent: no stock and no pending orders pass from one week to the next.
4. The cost of a plan over the period is its weekly cost (with the forecast demand) times $N$, plus $p$ for each pallet not served. The change in the shipping cost when a customer orders more or less than forecast is not counted.

### The orders

**What it shows.** The orders of C1 and C3 in weeks 1 to 13, in pallets. They are invented for the example. C2 and C4 order the same every week, by contract, so only C1 and C3 change.

**What to look at.** Week 13: C1 orders its forecast, 40 pallets, and C3 orders 46, as in section 6.

In [ ]:
# @title Orders of C1 and C3 { display-mode: "form" }
wa.show_orders()

### A plan week by week

**What it computes.** For the plan you choose, the *load* of each warehouse every week (the pallets of the customers it serves), its *margin* (capacity − load) and the pallets *not served* (those above capacity), all in pallets.

**What you can change.** The plan, A, B, M, D or E (the five feasible plans of section 4), and the weeks: weeks 1 to 12, or week 13.

**How to read it.** In the table, a negative margin is a load above capacity. In the figure, each line is the load of a warehouse week by week, the dashed line of the same colour is its capacity, and a red circle marks a week above capacity. Under it, a table sums up the five plans: the weeks above capacity in weeks 1 to 12, the smallest margin of each warehouse and the pallets not served in week 13.

**To try.** Which plans never go above capacity in weeks 1 to 12? Which of them still leave pallets unserved in week 13, and why?

In [ ]:
# @title A plan week by week { display-mode: "form" }
plan = "A"  # @param ["A", "B", "M", "D", "E"]
weeks = "1-12"  # @param ["1-12", "13"]
wa.plan_by_week(plan, weeks)

### The forecast error of C1 and C3

**What it computes.** For the customer you choose, the forecast of each week (the average of its four previous weeks, as in section 6), the error (orders − forecast) and four measures over weeks 5 to 12: MAE and bias in pallets/week, MAPE in %, and the *largest positive error*, the most the orders went above the forecast, $e^{+}_{\max} = \max_{t} e_t$. It also gives the forecast for week 13.

**What you can change.** The customer, C1 or C3, and a plan. If, in that plan, C1 and C3 are served from the same warehouse, their errors add up there: the cell gives that sum week by week.

**To try.** Which customer has the larger errors? In the plan where C1 and C3 share a warehouse, compare the largest sum of their errors with the spare capacity of that warehouse.

In [ ]:
# @title Forecast error of a customer { display-mode: "form" }
customer = "C1"  # @param ["C1", "C3"]
plan = "A"  # @param ["A", "B", "M", "D", "E"]
wa.customer_forecast(customer, plan)

### The cost of each plan over a period

**What it computes.** The cost of each feasible plan kept for $N$ weeks:

$$
\text{total cost} = N \cdot c + p \cdot U
$$

where $c$ is the weekly cost of the plan (€/week), $p$ the cost of a pallet that is not served (€/pallet) and $U$ the pallets not served in the period. In $s$ of the $N$ weeks there is a surprise: C1 and C3 order the pallets you choose. In the other weeks every customer orders its forecast, and every feasible plan serves all its pallets.

**What you can change.** $N$, $p$, $s$ (from 0 to $N$) and the orders of C1 and C3 in a week with a surprise.

**How to read it.** The table gives, for each plan, $c$, the pallets not served in one week with a surprise, $U$ and the total cost in €, from the cheapest plan to the most expensive. The bar chart shows the total costs, with the cheapest in green.

**To try.** Change $p$: from which value does plan A stop being the cheapest? Does that value change with $N$ or with $s$?

In [ ]:
# @title Cost of each plan over a period { display-mode: "form" }
N = 12  # @param {type:"slider", min:1, max:24, step:1}
p = 100  # @param {type:"slider", min:0, max:1000, step:10}
s = 1  # @param {type:"slider", min:0, max:24, step:1}
c1_order = 40  # @param {type:"slider", min:30, max:55, step:1}
c3_order = 46  # @param {type:"slider", min:25, max:55, step:1}
wa.period_costs(N, p, s, c1_order, c3_order)

## What to take away

- A model found a plan 145 €/week cheaper than the one diverted by hand, in a problem with only 16 plans.
- The number of plans grows so fast that real problems need optimisation methods (T2).
- With two objectives there is no single best plan, only a choice between Pareto-optimal plans (T3-T).
- A plan rests on a forecast. A large error can break it; a margin costs money and protects only where the spare room is. Re-planning every week is the usual answer (T3-P).
- Kept for several weeks, a plan meets orders that change: how much it costs depends on what a pallet that is not served costs, and on how often the orders go above the forecast.

All the numbers are explained in the [document of the example](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/warehouse_allocation/warehouse_allocation.md), which also has two scripts that reproduce them.